# GRT · Colab 레지스터 사후 분석

GPU 런타임(T4 권장)에서 현재 WikiText GRT checkpoint 하나를 FP32·단일 GPU로 분석합니다. 학습과 W&B 인증은 필요하지 않습니다. 학습의 world size와 관계없이 checkpoint를 분석할 수 있습니다.

Google Drive 또는 `/content`에 checkpoint와 원래 prepared 데이터의 다음 파일을 준비하세요.

```text
GRT_analysis/
├── checkpoints/run/best.pt
└── wikitext103_gpt2_bpe/
    ├── manifest.json
    ├── validation.pt
    └── tokenizer/
```

`last.pt`나 보존한 중간 checkpoint도 선택할 수 있습니다. prepared 데이터는 새로 만들지 않고 원본을 복사합니다. checkpoint manifest·validation/tokenizer hash가 다르면 실행을 중단합니다.

[사후 분석 사용 안내](https://github.com/YeoJune/GRT/blob/main/docs/workflow.md#posthoc-colab). 최신 main과 공통 `WikiTextPosthoc` helper를 사용합니다. 학습 notebook의 `job.report(posthoc=True, ...)`도 같은 분석을 실행합니다.


## 1. 코드·라이브러리 준비

최신 main과 분석 helper를 설치합니다. 같은 런타임에서 코드를 갱신한 뒤 이미 `grt`를 import했다면 런타임을 재시작하고 다시 실행하세요.


In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path('/content/GRT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch',
                    'https://github.com/YeoJune/GRT.git', str(repo)], check=True)
else:
    dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True)
    if dirty.strip():
        print('로컬 수정이 있어 코드 갱신을 생략합니다:', repo)
    else:
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)

# Editable 설치를 현재 Colab 커널에서도 찾도록 소스 경로를 등록합니다.
source_path = str(repo / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from grt.utils.posthoc import WikiTextPosthoc


## 2. Drive 연결

큰 checkpoint는 Drive에 업로드해 두면 세션이 바뀌어도 재사용할 수 있습니다. 왼쪽 파일 탭으로 `/content`에 직접 올리는 경우 `use_drive=False`로 바꾸고 다음 셀의 경로를 수정하세요.


In [ ]:
use_drive = True
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')


## 3. checkpoint·분석 옵션 선택

**M/MLP/backbone은 checkpoint에서 읽습니다.** 경로만 바꾸면 기존 reset/swap·retention·슬롯별 초기화와 새 분화 분석을 함께 실행합니다. 새 분석은 gate 원 logit·포화, state/update의 SVD·슬롯 간 CKA, 유형별 그룹 초기화·교환·복원, 같은 크기의 무작위 그룹, 초기화 비율별 손상 곡선을 포함합니다.

`window_starts=(0,16,32)`는 **각 validation stream 안의 segment 시작 위치**입니다. 각 위치에서 S0로 시작해 같은 warmup3 후8 segments를 측정하고 이후3 segments까지 추적합니다. 첫 window에서 closed/open/other(관측 내내<=.01 / >=.99 / 그 밖)를 정하고 **같은 슬롯 묶음**을 다른 두 window에서 검증합니다. 전체 warmup·측정·후속 범위가 겹치지 않아야 하며, batch를 바꾸면 stream 분할도 바뀝니다. 같은 M 비교에는 모든 분석 옵션을 동일하게 사용하세요.

기존 학습 중 RTLA와 맞춘 첫 window 결과는 유지합니다. `analyze_slots=True`는 첫 window에서만 단일 슬롯 기여를 측정합니다. 새 그룹 분석은 모든 window에서 수행하며 `group_repeats`는 무작위 대조군 반복 수입니다. 빠른 확인이 필요하면 슬롯 분석을 끄거나 무작위 반복을 줄일 수 있지만 비교 대상도 같은 조건으로 실행해야 합니다. 새 분화 분석 전체를 끄려면 `analyze_registers=False`로 바꿉니다.

`inspect()`는 calibration 전 forward 상한을 표시하고 실제 실행 시 중복 mask를 제거한 정확한 단계별 횟수를 출력합니다. gate 시간축은 고정 checkpoint의 probe이며 학습 과정의 변화가 아닙니다.


In [ ]:
from grt.utils.posthoc import WikiTextPosthoc

checkpoint = Path('/content/drive/MyDrive/GRT_analysis/checkpoints/run/best.pt')
prepared_dir = Path('/content/drive/MyDrive/GRT_analysis/wikitext103_gpt2_bpe')
output_dir = Path('/content/drive/MyDrive/GRT_analysis/outputs')

batch_size = 8
segments = 8
warmup_segments = 3
horizon = 4
analyze_slots = True       # 첫 window에서 기존 단일 슬롯 분석
analyze_registers = True   # 다중 window·logit·표현·그룹 개입
window_starts = (0, 16, 32)
group_repeats = 3

job = WikiTextPosthoc(
    checkpoint=checkpoint, prepared_dir=prepared_dir, output_dir=output_dir,
    device='cuda', batch_size=batch_size, segments=segments,
    warmup_segments=warmup_segments, horizon=horizon, slots=analyze_slots,
    registers=analyze_registers, window_starts=window_starts,
    group_repeats=group_repeats,
)
job.inspect()


## 4. 분석 실행

실제 checkpoint step·M·MLP·데이터 검증 결과와 예상 forward 횟수를 위 출력에서 확인하고 실행합니다. FP32, dropout off, gradient off, TF32 off입니다. 학습 checkpoint·학습 설정 파일을 수정하지 않습니다. 같은 입력·옵션·분석 코드의 완료 결과가 있으면 재사용합니다.

진행 횟수와 대략 남은 시간을 출력합니다. CUDA OOM이면 다른 GPU 프로그램을 종료하세요. batch를 줄여야 한다면 모든 비교 후보를 같은 새 batch로 다시 분석하세요. 세션이 중단돼도 Drive의 checkpoint는 유지되지만 미완료 분석은 처음부터 다시 실행합니다.


In [ ]:
result = job.run()


## 5. 그림 확인

- `memory.png`: 첫 window의 token 위치·segment별 reset/swap, 단일 슬롯 초기화, retention.
- `trace.png`: 첫 window의 batch 평균 write/state/update와 write pooling attention.
- `registers.png`: window별 gate 유형 수, sigmoid 전 logit, 문맥 평균을 뺀 state의 SVD 에너지, 평가 window의 무작위 그룹 손상 곡선.

상세 유형·threshold 민감도, CKA 행렬, 모든 그룹·lag·example의 paired ΔNLL과 크기를 맞춘 대조군·복원 효과는 JSON에 보존합니다. 미측정은 null입니다. raw state·원문 token·가중치 본체는 내보내지 않습니다.

세 유형은 write 동작의 관측 분류이며 기능의 증명이 아닙니다. CKA/rank·무작위 그룹 범위는 기술 통계이고 유의성 검정이 아닙니다. 그룹 효과는 합산하지 않으며 probe PPL은 전체 validation PPL을 대체하지 않습니다. read gate off 값을 읽기량으로, write pooling attention을 token→register attention으로 해석하지 않습니다.


In [ ]:
from IPython.display import Image, display

print('첫 window gate:', result['gate_statistics']['quantiles'])
if result['register_analysis'] is not None:
    for window in result['register_analysis']['windows']:
        counts = window['stability']['threshold_sensitivity']['0.01']['counts']
        print('window', window['start_segment'], window['role'], counts,
              '유형 유지율:', window['stability']['fraction_same_type'])
for name in ('memory.png', 'trace.png', 'registers.png'):
    image_path = job.analysis_dir / name
    if image_path.is_file():
        display(Image(filename=str(image_path)))


## 6. 보내줄 파일 하나 다운로드

**비교할 checkpoint마다 `posthoc_report.json` 하나만 보내주세요.** checkpoint hash·step, 원래 학습 설정, 데이터 manifest, 모든 분석 옵션·seed·window 위치·코드 hash·GPU/시간과 기존 RTLA·새 register 분석이 함께 들어갑니다. `register_analysis`에 window별 결과, calibration에서 정한 그룹, 동일한 개입 mask 목록, logit·표현 진단, reset/swap의 start×lag×example ΔNLL·token 수와 그룹 대조/복원 요약이 있습니다.

원본은 `results/posthoc/<report_id>.json`에 보존하고 [현재 연구 요약](https://github.com/YeoJune/GRT/blob/main/docs/research.md)에 비교를 반영합니다. 같은 조건·checkpoint step인지 확인한 뒤 유형 비율·포화·표현 중복·그룹 개입과 복구를 해석합니다. 완료된 실험 경과는 Markdown 아카이브에 보존합니다.

다운로드가 막히면 출력된 Drive 경로에서 JSON을 직접 받으세요. checkpoint·prepared 데이터는 계속 보관하되 보내지 않아도 됩니다. PNG·NPZ는 로컬 확인용이며 JSON 하나면 결과 정리가 가능합니다.


In [ ]:
job.report(download=True)
